In [1]:
import torch

In [2]:
torch.cuda.get_device_name(0)

'NVIDIA GeForce GTX 1080'

In [3]:
torch.cuda.is_available()

True

In [4]:
import torch


# Dataset 

In [5]:

import numpy as np
import glob
import time
import cv2



def load_file(filename):
    arrays = np.load(filename)
    return arrays

In [6]:
class MyDataset():
    def __init__(self, folds, path):
        self.folds = folds
        self.path = path
        with open(r'F:\VisualSpeech\labell.txt') as myfile:
            lines = myfile.read().splitlines()
        self.data_dir = [self.path + item for item in lines]
        self.data_files = glob.glob(self.path+'*/'+self.folds+'/*.npy')
        self.labels = {}
        for idx, label in enumerate(list(open(r'F:\VisualSpeech\labell.txt'))):
            label = label.strip()
            self.labels[label] = idx

    def __getitem__(self, idx):
        inputs = load_file(self.data_files[idx])
        labels = self.labels[self.data_files[idx].split('\\')[-1].split('_')[0]]
        #print("Label from file path:", labels)
        return inputs, labels

    def __len__(self):
        return len(self.data_files)


# cvtransforms

In [7]:

import random
import cv2
import numpy as np


def CenterCrop(batch_img, size):
    w, h = batch_img[0][0].shape[1], batch_img[0][0].shape[0]
    th, tw = size
    img = np.zeros((len(batch_img), len(batch_img[0]), th, tw))
    for i in range(len(batch_img)):
        x1 = int(round((w - tw))/2.)
        y1 = int(round((h - th))/2.)
        img[i] = batch_img[i, :, y1:y1+th, x1:x1+tw]
    return img

In [8]:
def RandomCrop(batch_img, size):
    w, h = batch_img[0][0].shape[1], batch_img[0][0].shape[0]
    th, tw = size
    img = np.zeros((len(batch_img), len(batch_img[0]), th, tw))
    for i in range(len(batch_img)):
        x1 = random.randint(0, 8)
        y1 = random.randint(0, 8)
        img[i] = batch_img[i, :, y1:y1+th, x1:x1+tw]
    return img

In [9]:

def HorizontalFlip(batch_img):
    for i in range(len(batch_img)):
        if random.random() > 0.5:
            for j in range(len(batch_img[i])):
                batch_img[i][j] = cv2.flip(batch_img[i][j], 1)
    return batch_img


def ColorNormalize(batch_img):
    mean = 0.413621
    std = 0.1700239
    batch_img = (batch_img - mean) / std
    return batch_img



# Model

In [10]:

import math
import numpy as np


import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable


def conv3x3(in_planes, out_planes, stride=1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                     padding=1, bias=False)

In [11]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super(BasicBlock, self).__init__()
        self.conv1 = conv3x3(inplanes, planes, stride)
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(planes, planes)
        self.bn2 = nn.BatchNorm2d(planes)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        if self.downsample is not None:
            residual = self.downsample(x)

        out += residual
        out = self.relu(out)

        return out

In [12]:
class ExceptionModel(nn.Module):

    def __init__(self, block, layers, num_classes=44):
        self.inplanes = 64
        super(ExceptionModel, self).__init__()
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)
        self.gloablavgpool = nn.AdaptiveAvgPool2d((1,1))
        self.bn = nn.BatchNorm1d(512)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()
            elif isinstance(m, nn.BatchNorm1d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes * block.expansion,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * block.expansion),
            )

        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample))
        self.inplanes = planes * block.expansion
        for i in range(1, blocks):
            layers.append(block(self.inplanes, planes))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.gloablavgpool(x)
        
        
        
        x= x.view(x.size(0),-1)
        x = self.bn(x)



        return x

In [13]:
class GRU(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes, every_frame=True):
        super(GRU, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.every_frame = every_frame
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True, bidirectional=True, dropout=0.2)
        self.fc = nn.Linear(hidden_size*2, num_classes)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        self.gru.flatten_parameters()
        h0 = Variable(torch.zeros(self.num_layers*2, x.size(0), self.hidden_size))
        out, _ = self.gru(x, h0)
        
        out = self.dropout(out).mean(1)
        out = self.fc(out)

        return out

In [14]:
class Lipreading(nn.Module):
    def __init__(self, mode, inputDim=256, hiddenDim=1024, nClasses=44, frameLen=30, every_frame=True):
        super(Lipreading, self).__init__()
        self.mode = mode
        self.inputDim = inputDim
        self.hiddenDim = hiddenDim
        self.nClasses = nClasses
        self.frameLen = 30
        self.every_frame = every_frame
        self.nLayers = 3
        # frontend3D
        self.frontend3D = nn.Sequential(
                nn.Conv3d(1, 64, kernel_size=(5, 7, 7), stride=(1, 2, 2), padding=(2, 3, 3), bias=False),
                nn.BatchNorm3d(64),
                nn.ReLU(True),
                nn.MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1))
                )
        

        self.ExceptionModel = ExceptionModel(BasicBlock, [2, 2, 2, 2], num_classes=self.inputDim)
        self.dropout = nn.Dropout(0.2)



        self.gru = GRU(self.inputDim, self.hiddenDim, self.nLayers, self.nClasses, self.every_frame)
            # initialize
        self._initialize_weights()

    def forward(self, x):
        x = self.frontend3D(x)
        x = x.transpose(1, 2)
        x = x.contiguous()
        x = x.view(-1, 64, x.size(3), x.size(4))
        x = self.ExceptionModel(x)

        x = self.dropout(x)

        x = x.view(-1, self.frameLen, self.inputDim)
        x = self.gru(x)

        return x

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                n = m.kernel_size[0] * m.kernel_size[1] * m.kernel_size[2] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
                if m.bias is not None:
                    m.bias.data.zero_()

            elif isinstance(m, nn.Conv2d):
                n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
                if m.bias is not None:
                    m.bias.data.zero_()

            elif isinstance(m, nn.Conv1d):
                n = m.kernel_size[0] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
                if m.bias is not None:
                    m.bias.data.zero_()

            elif isinstance(m, nn.BatchNorm3d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()

            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()

            elif isinstance(m, nn.BatchNorm1d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()

In [15]:

!pip install torchsummary

In [16]:

if (__name__ == '__main__'):
    import torchsummary
    model = ExceptionModel(BasicBlock, [3,4,6,3], num_classes=44)
    #torchsummary.summary(model,(64,22,22),device='cpu')

# Main

In [17]:
import os
import time
import random
import logging
import argparse
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

#from model import *
#from dataset import *
#from cvtransforms import *

SEED = 1
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
np.random.seed(SEED)

In [18]:
import numpy as np
import torch
import torchvision
from torchvision import datasets, models, transforms
import torch.utils.data as data
#from torch.utils.tensorboard import SummaryWriter
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
#from nets import *
import time, os, copy, argparse
import multiprocessing
from torchsummary import summary
#from matplotlib import pyplot as plt

C:\Users\ml\.conda\envs\py39\lib\site-packages\torchvision\io\image.py:13: UserWarning: Failed to load image Python extension: '[WinError 127] The specified procedure could not be found'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [19]:
parser = argparse.ArgumentParser(description='Pytorch-GLMIM-LRW')
parser.add_argument('--nClasses', default=44, type=int, help='the number of classes')
#path : path to model, when you start Baseline, it's empty, otherwise it would load the model.
parser.add_argument('--path', default=r'', type=str, help='path to Baseline, empty for training')
parser.add_argument('--dataset', default=r'F:/VisualSpeech/ZhaoNewDataset/', type=str, help='path/to/lrw/roi_80_116_175_211_npy_gray/')
parser.add_argument('--mode', default='Baseline', type=str)
parser.add_argument('--every-frame', default=True, action='store_false', help='predicition based on every frame')
parser.add_argument('--lr', default=1e-4, type=float, help='initial learning rate')
parser.add_argument('--batch-size', default=32, type=int, help='mini-batch size')
#parser.add_argument('--workers', default=5, type=int, help='number of data loading workers')
parser.add_argument('--epochs', default=35, type=int, help='number of total epochs')
parser.add_argument('--s_epochs', default=1, type=int, help='number of start epochs')
parser.add_argument('--interval', default=40, type=int, help='display interval')
parser.add_argument('--test', default=False, action='store_false', help='perform on the test phase')


_StoreFalseAction(option_strings=['--test'], dest='test', nargs=0, const=False, default=False, type=None, choices=None, required=False, help='perform on the test phase', metavar=None)

In [20]:

args = parser.parse_args([])


In [21]:
torch.cuda.is_available()
if torch.cuda.is_available():
    torch.set_default_tensor_type('torch.cuda.FloatTensor')

C:\Users\ml\.conda\envs\py39\lib\site-packages\torch\__init__.py:614: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at C:\cb\pytorch_1000000000000\work\torch\csrc\tensor\python_tensor.cpp:453.)
  _C._set_default_tensor_type(t)


In [22]:
def data_loader(args):
    dsets = {x: MyDataset(x, args.dataset) for x in ['train','val', 'test']}
    dset_loaders = {x: DataLoader(dsets[x], batch_size=args.batch_size, shuffle=True,generator=torch.Generator(device="cuda")) for x in ['train', 'val', 'test']}
    dset_sizes = {x: len(dsets[x]) for x in ['train', 'val', 'test']}
    ##########################################
    #print('\nStatistics: train: {}, val: {}, test: {}'.format(dset_sizes['train'], dset_sizes['val'], dset_sizes['test']))
    return dset_loaders, dset_sizes

In [23]:


def reload_model(model, logger, path=""):
    if not bool(path):
        logger.info('train from scratch')
        return
    own_state = model.state_dict()
    state_dict = torch.load(path)
    for name, param in state_dict.items():
        if name not in own_state:
            print('layer {} skip, not exist'.format(name))
            continue
        if isinstance(param, nn.Parameter):
            param = param.data
        if own_state[name].shape != param.shape:
            print('layer {} skip, shape not same'.format(name))
            continue
        own_state[name].copy_(param)

In [24]:
def showLR(optimizer):
    lr = []
    for param_group in optimizer.param_groups:
        lr += [param_group['lr']]
    return lr

<h1> new test, train, test_adam

In [ ]:
import time
import numpy as np


def train(model, dset_loaders, criterion, epoch, phase, optimizer, args, logger, use_gpu):
    model.train()
    logger.info('-' * 2)
    logger.info('Epoch {}/{}'.format(epoch, args.epochs - 1))
    logger.info('Current Learning rate: {}'.format(showLR(optimizer)))

    running_loss, running_corrects, running_all = 0., 0., 0.
    since = time.time()
    last_time_batch_idx = -1
    for batch_idx, (inputs, targets) in enumerate(dset_loaders[phase]):
        batch_img = RandomCrop(inputs.cpu().numpy(), (90, 90))
        batch_img = ColorNormalize(batch_img)
        batch_img = HorizontalFlip(batch_img)

        batch_img = np.reshape(batch_img, (batch_img.shape[0], batch_img.shape[1], batch_img.shape[2], batch_img.shape[3], 1))
        inputs = torch.from_numpy(batch_img)
        inputs = inputs.float().permute(0, 4, 1, 2, 3).contiguous()
        if use_gpu:
            inputs = inputs.cuda()
            targets = targets.cuda()
        optimizer.zero_grad()
        outputs = model(inputs)
        _, preds = torch.max(outputs.data, 1)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        # statistics
        running_loss += loss.item() * inputs.size(0)
        batch_correct = (preds == targets.data).sum().item()
        running_corrects += batch_correct
        running_all += len(inputs)

        if batch_idx % args.interval == 0 or (batch_idx == len(dset_loaders[phase])-1):
            logger.info('Process:  [({:.0f}%)]\tLoss batch: {:.4f}\tLoss total: {:.4f}\tAcc batch:{:.4f}\tAcc total:{:.4f}\tEstimated time:{:5.0f}s'.format(
             
                100. * batch_idx / (len(dset_loaders[phase])-1),
                float(loss),
                float(running_loss) / running_all,
                float(batch_correct) / len(inputs),
                float(running_corrects) / running_all,
                (time.time() - since) / (batch_idx - last_time_batch_idx) * (len(dset_loaders[phase]) - batch_idx - 1))),
            last_time_batch_idx = batch_idx
            since = time.time()

    loss_epoch =  float(running_loss) / len(dset_loaders[phase].dataset)
    acc_epoch = float(running_corrects) / len(dset_loaders[phase].dataset)

    logger.info('{} Epoch:\t{:2}\tLoss: {:.4f}\tAcc:{:.4f}\n'.format(
        phase,
        epoch,
        loss_epoch,
        acc_epoch))
    
    return loss_epoch, acc_epoch

In [ ]:
def test2(model, dset_loaders, criterion, epoch, phase, args, logger, use_gpu, save=True):
    model.eval()
    with torch.no_grad():
        running_loss, running_corrects, running_all = 0., 0., 0.
        since = time.time()
        last_time_batch_idx = -1
        all_preds = []
        all_labels = []
        for batch_idx, (inputs, targets) in enumerate(dset_loaders[phase]):
            batch_img = CenterCrop(inputs.cpu().numpy(), (90, 90))
            batch_img = ColorNormalize(batch_img)

            batch_img = np.reshape(batch_img, (batch_img.shape[0], batch_img.shape[1], batch_img.shape[2], batch_img.shape[3], 1))
            inputs = torch.from_numpy(batch_img)
            inputs = inputs.float().permute(0, 4, 1, 2, 3).contiguous()
            if use_gpu:
                inputs = inputs.cuda()
                targets = targets.cuda()
            outputs = model(inputs)

            _, preds = torch.max(outputs.data, 1)
            loss = criterion(outputs, targets)
            # statistics
            running_loss += loss.item() * inputs.size(0)
            running_corrects += (preds == targets.data).sum().item()
            running_all += len(inputs)

            if batch_idx % args.interval == 0 or (batch_idx == len(dset_loaders[phase])-1):
                logger.info('Process: [({:.0f}%)]\tLoss: {:.4f}\tAcc:{:.4f}\tEstimated time:{:5.0f}s'.format(
                    
                    100. * batch_idx / (len(dset_loaders[phase])-1),
                    float(running_loss) / running_all,
                    float(running_corrects) / running_all,
                    (time.time() - since)/(batch_idx-last_time_batch_idx) * (len(dset_loaders[phase]) - batch_idx - 1))),
                last_time_batch_idx = batch_idx
                since = time.time()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(targets.cpu().numpy())

        loss_epoch = float(running_loss) / len(dset_loaders[phase].dataset)
        acc_epoch = float(running_corrects) / len(dset_loaders[phase].dataset)
        logger.info('{} Epoch:\t{:2}\tLoss: {:.4f}\tAcc:{:.4f}'.format(
            phase,
            epoch,
            loss_epoch,
            acc_epoch)+'\n')

    if save:
        # Returning test accuracy and confusion matrix along with other necessary values
        return loss_epoch, acc_epoch, all_preds, all_labels
    else:
        return None, None, None, None  # Return placeholders for consistency


In [ ]:
def test(model, dset_loaders, criterion, epoch, phase, args, logger, use_gpu, save=True):
    model.eval()
    with torch.no_grad():
        running_loss, running_corrects, running_all = 0., 0., 0.
        since = time.time()
        last_time_batch_idx = -1
        all_preds = []
        all_labels = []
        for batch_idx, (inputs, targets) in enumerate(dset_loaders[phase]):
            batch_img = CenterCrop(inputs.cpu().numpy(), (90, 90))
            batch_img = ColorNormalize(batch_img)

            batch_img = np.reshape(batch_img, (batch_img.shape[0], batch_img.shape[1], batch_img.shape[2], batch_img.shape[3], 1))
            inputs = torch.from_numpy(batch_img)
            inputs = inputs.float().permute(0, 4, 1, 2, 3).contiguous()
            if use_gpu:
                inputs = inputs.cuda()
                targets = targets.cuda()
            outputs = model(inputs)

            _, preds = torch.max(outputs.data, 1)
            loss = criterion(outputs, targets)
            # statistics
            running_loss += loss.item() * inputs.size(0)
            running_corrects += (preds == targets.data).sum().item()
            running_all += len(inputs)

            if batch_idx % args.interval == 0 or (batch_idx == len(dset_loaders[phase])-1):
                logger.info('Process:  [({:.0f}%)]\tLoss: {:.4f}\tAcc:{:.4f}\tEstimated time:{:5.0f}s'.format(
                   
                    100. * batch_idx / (len(dset_loaders[phase])-1),
                    float(running_loss) / running_all,
                    float(running_corrects) / running_all,
                    (time.time() - since)/(batch_idx-last_time_batch_idx) * (len(dset_loaders[phase]) - batch_idx - 1))),
                last_time_batch_idx = batch_idx
                since = time.time()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(targets.cpu().numpy())

        loss_epoch = float(running_loss) / len(dset_loaders[phase].dataset)
        acc_epoch = float(running_corrects) / len(dset_loaders[phase].dataset)
        logger.info('{} Epoch:\t{:2}\tLoss: {:.4f}\tAcc:{:.4f}'.format(
            phase,
            epoch,
            loss_epoch,
            acc_epoch)+'\n')

    if save:
        # Returning test accuracy and confusion matrix along with other necessary values
        return loss_epoch, acc_epoch
    else:
        return None, None, None, None  # Return placeholders for consistency


In [ ]:
import os
import logging
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

def test_adam(args, use_gpu):
    save_path = './' + args.mode
    if not os.path.isdir(save_path):
        os.mkdir(save_path)
    
    # Logging setup
    filename = save_path + '/' + args.mode + '_' + str(args.lr) + '.txt'
    logger_name = "mylog"
    logger = logging.getLogger(logger_name)
    logger.setLevel(logging.INFO)
    fh = logging.FileHandler(filename, mode='a')
    fh.setLevel(logging.INFO)
    logger.addHandler(fh)
    console = logging.StreamHandler()
    console.setLevel(logging.INFO)
    logger.addHandler(console)

    # Model initialization and setup
    net = Lipreading(mode=args.mode, inputDim=512, hiddenDim=1024, nClasses=args.nClasses, frameLen=30, every_frame=args.every_frame)
    print(net)
    reload_model(net, logger, args.path)
    model = torch.nn.DataParallel(net)
    if use_gpu:
        model = model.cuda()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(net.parameters(), lr=args.lr, weight_decay=5e-5, amsgrad=True)
    dset_loaders, dset_sizes = data_loader(args)

    train_losses = []
    val_losses = []
    train_accuracies = []
    val_accuracies = []
    best_val_acc = -1
    best_val_epoch = None
    best_test_acc = -1
    best_test_epoch = None
    best_confusion_matrix = None

    for epoch in range(args.s_epochs, args.epochs):
        train_loss, train_acc = train(model, dset_loaders, criterion, epoch, 'train', optimizer, args, logger, use_gpu)
        val_loss, val_acc = test(model, dset_loaders, criterion, epoch, 'val', args, logger, use_gpu, True)
        
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accuracies.append(train_acc)
        val_accuracies.append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_val_epoch = epoch

            state_dict = net.state_dict()

            _,best_test_acc, best_test_predictions, best_test_labels = test2(model, dset_loaders, criterion, epoch, 'test', args, logger, use_gpu, True)
            best_confusion_matrix = confusion_matrix(best_test_labels, best_test_predictions)
            best_test_epoch = epoch

    torch.save(state_dict, 'modelnew44.pth')
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(train_accuracies, label='Train Accuracy')
    plt.plot(val_accuracies, label='Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.show()

    print("Best Validation Epoch:", best_val_epoch, "with Validation Accuracy:", best_val_acc)
    print("Best Test Epoch:", best_test_epoch, "with Test Accuracy:", best_test_acc)
    print("Confusion Matrix of the Best Model:\n", best_confusion_matrix)
    
    # Compute precision, recall, and F1 score
    precision = precision_score(best_test_labels, best_test_predictions, average='weighted')
    recall = recall_score(best_test_labels, best_test_predictions, average='weighted')
    f1 = f1_score(best_test_labels, best_test_predictions, average='weighted')

    # Print precision, recall, and F1 score
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1 Score:", f1)
    
    # Plot confusion matrix
    sns.heatmap(best_confusion_matrix, annot=True)
    plt.xlabel('Predicted labels')
    plt.ylabel('True labels')
    plt.show()
    
    return best_test_acc, best_confusion_matrix


<h1>previous results</h1>

In [ ]:
if __name__ == '__main__':
    use_gpu = torch.cuda.is_available()
    test_adam(args, use_gpu)
    

<h1>output handling

In [26]:
#my frames selection code
import cv2
import os
import shutil
def extract_keyframes(video_path, output_folder):
   
    video_capture = cv2.VideoCapture(video_path)
    total_frames = int(video_capture.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Total frames in the video: {total_frames}")

    selected_keyframes = []


    num_keyframes = 30

    if total_frames > num_keyframes:
     
        video_capture = cv2.VideoCapture(video_path)

        frame_skip = total_frames // num_keyframes

        selected_frames_count = 0

        for i in range(total_frames):
            ret, frame = video_capture.read()

            if not ret:
                break

            # Select frames based on frame skip value
            if i % frame_skip == 0 and selected_frames_count < num_keyframes:
                selected_keyframes.append(frame)
                selected_frames_count += 1

        video_capture.release()

    
        print(f"Number of frames selected: {len(selected_keyframes)}")
        # Save the selected keyframes as a video
        video_filename = os.path.splitext(os.path.basename(video_path))[0]
        output_video_path = os.path.join(output_folder, f'{video_filename}_output_video.mp4')
        height, width, _ = selected_keyframes[0].shape
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        video_writer = cv2.VideoWriter(output_video_path, fourcc, 30, (width, height))

        for keyframe in selected_keyframes:
            video_writer.write(keyframe)

        video_writer.release()

        print(f"Output video saved as: {output_video_path}")

    else:
        print("Not enough frames in the video for keyframe selection, so directly saved")
        video_filename = os.path.basename(video_path)
        output_video_path = os.path.join(output_folder, video_filename)
        shutil.copyfile(video_path, output_video_path)

# Input video
#input_video_path = r"C:\Users\ml\Desktop\21DataNew\AllVideosof100Classes\newAll\Bayu01_0018.mp4"

# Output folder 
#output_folder_path = r"F:\vid\newVid"





In [39]:
#my code for marked frames+ landmarks
import dlib
import cv2
import numpy as np
import os

VIDEO_EXTENSIONS = ('.mp4', '.avi', '.mkv')
#INPUT_FOLDER = r"F:\vid\newVid"  # Input folder containing video files
OUTPUT_FOLDER = r"F:\vid\npy"  # Output folder for NumPy arrays
cropped_frames_folder=  r'F:\vid\frames'

RESIZE_DIM = (96, 96) 
LIP_MARGIN = 0.4 

face_detector = dlib.get_frontal_face_detector()
lip_landmark_predictor = dlib.shape_predictor(r"F:\VisualSpeech\shape_predictor_68_face_landmarks.dat")
def mark_crop_resize_gray(frame):
    faces = face_detector(frame)
    
    if len(faces) == 0:
        return None, None
    
    face = faces[0]  
    lip_landmarks = lip_landmark_predictor(frame, face)
    landmark = [(lip_landmarks.part(i).x, lip_landmarks.part(i).y) for i in range(48, 68)]

    marked_frame = frame.copy()

    for i in range(48, 68):
        x, y = lip_landmarks.part(i).x, lip_landmarks.part(i).y
        marked_frame[y - 3:y + 3+ 1, x - 1:x + 2] = (0, 0, 0)
        marked_frame[y - 1:y + 2, x - 3:x + 3 + 1] = (0, 0, 0)
    
    x, y, w, h = face.left(), face.top(), face.width(), face.height()
    roi_x = x + w // 4
    roi_y = y + h // 2
    roi_size = 256
    mouth_roi = marked_frame[roi_y:roi_y + roi_size, roi_x:roi_x + roi_size]

    mouth_roi_gray = cv2.cvtColor(mouth_roi, cv2.COLOR_BGR2GRAY)

    mouth_roi_gray_resized = cv2.resize(mouth_roi_gray, (98, 98))

    avg_x = np.mean([x for x, y in landmark])
    avg_y = np.mean([y for x, y in landmark])
    normalized_landmark = [((x - avg_x) / avg_x, (y - avg_y) / avg_y) for x, y in landmark]
    padded_normalized_landmark = np.pad(normalized_landmark, [(0, 98 - len(normalized_landmark)), (0, 0)], mode='constant') 
    
    return marked_frame, mouth_roi_gray_resized, padded_normalized_landmark

def preprocess_video(INPUT_FOLDER,OUTPUT_FOLDER,idxx):
    OUTPUT_FOLDER = r"F:\vid\npy"
    for video_file in os.listdir(INPUT_FOLDER):
        video_path = os.path.join(INPUT_FOLDER, video_file)

        if video_file.lower().endswith(VIDEO_EXTENSIONS):
            print(f"Processing video: {video_path}")
            try:
               OUTPUT_FOLDER = r"F:\vid\npy" 
               video_array= preprocess_video(video_path, OUTPUT_FOLDER, idxx)
            except Exception as e:
                print(f"Error processing video {video_file}: {str(e)}")
                continue        
    cap = cv2.VideoCapture(video_path)
    frames_with_landmarks = []
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        marked_frame, mouth_roi_gray_resized, normalized_landmark = mark_crop_resize_gray(frame)

        if marked_frame is not None:
    
            frame_with_normalized_landmarks = np.hstack((mouth_roi_gray_resized, normalized_landmark))
            frame_with_zeros = np.vstack((frame_with_normalized_landmarks, np.zeros((2, 100))))  # Add two lines of zeros
            frames_with_landmarks.append(frame_with_zeros)
    clear_folder(cropped_frames_folder)    
    for idx, frame_landmarks in enumerate(frames_with_landmarks):
        cropped_frame = frame_landmarks[:, :RESIZE_DIM[0]]  # Extract the cropped frame
        output_filename = f"cropped_frame_{idxx}.png"
        output_path = os.path.join(cropped_frames_folder, output_filename)
        cv2.imwrite(output_path, cropped_frame)
    
        idxx= idxx+1    

    cap.release()
    video_array = np.array(frames_with_landmarks)
    print("Shape of the video array:", video_array.shape)


    video_name = os.path.splitext(os.path.basename(video_path))[0]
    output_folder = r"F:\vid\npy"
    output_file = os.path.join(output_folder, f"{video_name}.npy")
    np.save(output_file, video_array)
    return video_array




In [40]:
#currently using this for adding frames and gettingsame size numpy arrray
import os
import numpy as np

def again_resize():
    numpy_folder = r'F:\vid\npy'

#  expected size
    expected_size = (30, 100, 100)

    npy_files = [f for f in os.listdir(numpy_folder) if f.endswith('.npy')]

# Sort the files to ensure they are processed in order
    npy_files.sort()


    counter = 1


    for npy_file in npy_files:
        npy_path = os.path.join(numpy_folder, npy_file)

 
        loaded_array = np.load(npy_path)

        if loaded_array.shape == expected_size:
            print(f"File {npy_file} has the correct size. Keeping it.")
        else:
            num_frames, frame_height, frame_width = loaded_array.shape
            if num_frames < 30:
            # Duplicate frames as needed
                frames_to_duplicate = 30 - num_frames
                duplicated_frames = np.repeat(loaded_array, frames_to_duplicate, axis=0)
                new_array = np.vstack((duplicated_frames, loaded_array))
                new_array = new_array[:30]  # Keep the first 30 frames

                np.save(npy_path, new_array)
                print(f"File {npy_file} had fewer frames. Duplicated frames to reach 30.")
            elif num_frames > 30:
                new_array = loaded_array[-30:]
                np.save(npy_path, new_array)
                print(f"File {npy_file} had more frames. Discarded extra frames from the beginning.")

    # Rename the file with a numerical sequence
        new_name = f" {npy_file}"
        new_path = os.path.join(numpy_folder, new_name)
        os.rename(npy_path, new_path)
    
    # Increment the counter

        counter += 1
    print("done")

In [41]:
class_names = [
    "BARI", "BAYU", "BEBOHAR", "BONDHO", "CHOKH", "DHONNOBAD", "EKA", "EKMOT", "GACH",
    "GORBO", "KANNA", "KENO", "LOKKHO", "MARA", "MASH", "MATI", "MULLO", "NAM", "NICHE",
    "OBOSSHOI", "ODHIKHAR", "ONDHOKAR", "ONUMOTI", "OPORADH", "PAKHI", "PORISHKAR", 
    "POROBORTI", "PRODORSHON", "PROTARONA", "PROYOJON", "PURATON", "ROKTO", "SESH", 
    "SHANTI", "SHOB", "SHOCHETON", "SHOITAN", "SOTTO", "TAKA", "TARPOR", "TINI", 
    "UPOHAR", "UPOVOG", "VALOBASHA"
]

def preprocess_input(input_array):
     
    
  
    th, tw = 90, 90  # Desired crop size
    if len(input_array.shape) == 3:
        h, w = input_array.shape[1], input_array.shape[2]
        x1 = int(round((w - tw) / 2))
        y1 = int(round((h - th) / 2))
        input_array = input_array[:, y1:y1+th, x1:x1+tw]
    elif len(input_array.shape) == 2:
        h, w = input_array.shape[0], input_array.shape[1]
        x1 = int(round((w - tw) / 2))
        y1 = int(round((h - th) / 2))
        input_array = input_array[y1:y1+th, x1:x1+tw]
    input_array = ColorNormalize(input_array)

    input_array = np.expand_dims(input_array, axis=0)  # Add batch dimension
    input_array = np.expand_dims(input_array, axis=0)  # Add frames dimension

    preprocessed_input = torch.from_numpy(input_array).float()
    return preprocessed_input

def ColorNormalize(batch_img):
    mean = 0.413621
    std = 0.1700239
    batch_img = (batch_img - mean) / std
    return batch_img
    
def predict_single_output(model, test_data, use_gpu=False):
    model.eval()  # Set the model to evaluation mode
    with torch.no_grad():
        if use_gpu:
            test_data = test_data.cuda()
        output = model(test_data)
        _, predicted_label = torch.max(output, 1)
    return predicted_label.item()  



In [42]:
C:\Users\ml\Desktop\21DataNew\AllVideosof100Classes\newAll

SyntaxError: unexpected character after line continuation character (708950573.py, line 1)

In [43]:
def clear_folder(folder_path):
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        try:
            if os.path.isfile(file_path):
                os.unlink(file_path)
        except Exception as e:
            print(f"Failed to delete {file_path}. Reason: {e}")

In [ ]:
import gradio as gr


In [44]:
import gradio as gr
from IPython.display import display, HTML

# Define a function to process the video input
def process_video(video):

    
    output_folder= r"F:\vid\npy"
    #input_video_path = video.name 
    #input_video_path = r"C:\Users\ml\Desktop\21DataNew\AllVideosof100Classes\newAll\Bayu01_0018.mp4"  # Gradio automatically saves the uploaded file with a temporary name
    clear_folder(output_folder)
    output_folder_path =  r"F:\vid\newVid"
    output_folder=r"F:\vid\npy"
    clear_folder(output_folder_path)
    extract_keyframes(video, output_folder_path)
    preprocess_video(output_folder_path,output_folder, 0)
    again_resize()
    saved_model = Lipreading(mode=args.mode, inputDim=512, hiddenDim=1024, nClasses=44, frameLen=30, every_frame=args.every_frame)
    saved_model.load_state_dict(torch.load('modelnew44.pth'))

    # Check if GPU is available
    use_gpu = torch.cuda.is_available()
    input_file = os.listdir(output_folder)[0]  # Assuming only one file is present in the folder
    input_data = np.load(os.path.join(output_folder, input_file))
    input_data = preprocess_input(input_data)

    # Predict the class
    predicted_class = predict_single_output(saved_model, input_data, use_gpu)

    # Return the predicted class
    return class_names[predicted_class]
    

            
video_input_html = """
<label for="video">Upload Video:</label><br>

"""

# Display video input HTML
display(HTML(video_input_html))

# Create the Gradio interface
interface = gr.Interface(fn=process_video, inputs=["video"], outputs="text", title="Bangla VSR")
interface.launch()

Running on local URL:  http://127.0.0.1:7881

To create a public link, set `share=True` in `launch()`.


In [45]:
import gradio as gr
from IPython.display import display, HTML

# Define a function to process the video input
def process_video(video):

    
    output_folder= r"F:\vid\npy"
    #input_video_path = video.name 
    #input_video_path = r"C:\Users\ml\Desktop\21DataNew\AllVideosof100Classes\newAll\Bayu01_0018.mp4"  # Gradio automatically saves the uploaded file with a temporary name
    clear_folder(output_folder)
    output_folder_path =  r"F:\vid\newVid"
    output_folder=r"F:\vid\npy"
    clear_folder(output_folder_path)
    extract_keyframes(video, output_folder_path)
    preprocess_video(output_folder_path,output_folder, 0)
    again_resize()
    saved_model = Lipreading(mode=args.mode, inputDim=512, hiddenDim=1024, nClasses=44, frameLen=30, every_frame=args.every_frame)
    saved_model.load_state_dict(torch.load('modelnew44.pth'))

    # Check if GPU is available
    use_gpu = torch.cuda.is_available()
    input_file = os.listdir(output_folder)[0]  # Assuming only one file is present in the folder
    input_data = np.load(os.path.join(output_folder, input_file))
    input_data = preprocess_input(input_data)

    # Predict the class
    predicted_class = predict_single_output(saved_model, input_data, use_gpu)

    # Return the predicted class
    return class_names[predicted_class]
    

            
video_input_html = """
<label for="video">Upload Video:</label><br>

"""

# Display video input HTML
display(HTML(video_input_html))

# Create the Gradio interface
interface = gr.Interface(fn=process_video, inputs=["file"], outputs="text", title="Bangla VSR file")
interface.launch()

Running on local URL:  http://127.0.0.1:7882

To create a public link, set `share=True` in `launch()`.


Total frames in the video: 30
Not enough frames in the video for keyframe selection, so directly saved
Processing video: F:\vid\newVid\aka04_009.mp4
Error processing video aka04_009.mp4: [WinError 267] The directory name is invalid: 'F:\\vid\\newVid\\aka04_009.mp4'
Shape of the video array: (30, 100, 100)
File aka04_009.npy has the correct size. Keeping it.
done
Total frames in the video: 30
Not enough frames in the video for keyframe selection, so directly saved
Processing video: F:\vid\newVid\Bari01_0011.mp4
Error processing video Bari01_0011.mp4: [WinError 267] The directory name is invalid: 'F:\\vid\\newVid\\Bari01_0011.mp4'
Shape of the video array: (30, 100, 100)
File Bari01_0011.npy has the correct size. Keeping it.
done
Total frames in the video: 24
Not enough frames in the video for keyframe selection, so directly saved
Processing video: F:\vid\newVid\Bayu05_0011.mp4
Error processing video Bayu05_0011.mp4: [WinError 267] The directory name is invalid: 'F:\\vid\\newVid\\Bayu05_

In [35]:
import tkinter as tk
from tkinter import filedialog
import cv2
from PIL import Image, ImageTk

# Define your function to process the video here
def process_video(video_path):
    # Your video processing code here
    output_folder= r"F:\vid\npy"
    #input_video_path = video.name 
    #input_video_path = r"C:\Users\ml\Desktop\21DataNew\AllVideosof100Classes\newAll\Bayu01_0018.mp4"  # Gradio automatically saves the uploaded file with a temporary name
    clear_folder(output_folder)
    output_folder_path =  r"F:\vid\newVid"
    output_folder=r"F:\vid\npy"
    clear_folder(output_folder_path)
    extract_keyframes(video_path, output_folder_path)
    preprocess_video(output_folder_path,output_folder, 0)
    again_resize()
    saved_model = Lipreading(mode=args.mode, inputDim=512, hiddenDim=1024, nClasses=44, frameLen=30, every_frame=args.every_frame)
    saved_model.load_state_dict(torch.load('modelnew44.pth'))

    # Check if GPU is available
    use_gpu = torch.cuda.is_available()
    input_file = os.listdir(output_folder)[0]  # Assuming only one file is present in the folder
    input_data = np.load(os.path.join(output_folder, input_file))
    input_data = preprocess_input(input_data)

    # Predict the class
    predicted_class = predict_single_output(saved_model, input_data, use_gpu)

    # Return the predicted class
    return class_names[predicted_class]

def upload_file():
    filename = filedialog.askopenfilename()
    result_label.config(text="Processing...")
    result = process_video(filename)
    result_label.config(text=result)

def display_video():
    filename = filedialog.askopenfilename()
    cap = cv2.VideoCapture(filename)
    while cap.isOpened():
        ret, frame = cap.read()
        if ret:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img = ImageTk.PhotoImage(Image.fromarray(frame))
            video_label.config(image=img)
            video_label.image = img
            video_label.update()
        else:
            break
    cap.release()

# Create the main Tkinter window
root = tk.Tk()
root.title("Video Processing App")

# Create a button to upload a video file
upload_button = tk.Button(root, text="Upload Video", command=upload_file)
upload_button.pack(pady=10)

# Create a label to display the result
result_label = tk.Label(root, text="", wraplength=400)
result_label.pack(pady=10)

# Create a frame for video display
video_frame = tk.Frame(root)
video_frame.pack(pady=10)

# Create a label to display the video
video_label = tk.Label(video_frame)
video_label.pack()

root.mainloop()

Total frames in the video: 30
Not enough frames in the video for keyframe selection, so directly saved
Processing video: F:\vid\newVid\aka04_0002.mp4
Error processing video aka04_0002.mp4: [WinError 267] The directory name is invalid: 'F:\\vid\\newVid\\aka04_0002.mp4'
Shape of the video array: (30, 100, 100)
File aka04_0002.npy has the correct size. Keeping it.
done
Total frames in the video: 30
Not enough frames in the video for keyframe selection, so directly saved
Processing video: F:\vid\newVid\bari04_014.mp4
Error processing video bari04_014.mp4: [WinError 267] The directory name is invalid: 'F:\\vid\\newVid\\bari04_014.mp4'
Shape of the video array: (30, 100, 100)
File bari04_014.npy has the correct size. Keeping it.
done


<h1> cache clear

In [ ]:
import gc
import torch
gc.collect()
torch.cuda.empty_cache()

In [ ]:
del model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import numpy as np
import torch

import numpy as np
import torch

def preprocess_input(input_data):
    input_array = np.load(input_data)
    
    # Assuming input_array.shape is (frames, height, width) or (height, width)
    th, tw = 90, 90  # Desired crop size
    if len(input_array.shape) == 3:
        # For (frames, height, width) array
        h, w = input_array.shape[1], input_array.shape[2]
        x1 = int(round((w - tw) / 2))
        y1 = int(round((h - th) / 2))
        input_array = input_array[:, y1:y1+th, x1:x1+tw]
    elif len(input_array.shape) == 2:
        # For (height, width) array
        h, w = input_array.shape[0], input_array.shape[1]
        x1 = int(round((w - tw) / 2))
        y1 = int(round((h - th) / 2))
        input_array = input_array[y1:y1+th, x1:x1+tw]

    # Convert to grayscale
    #input_array = np.array([cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) for frame in input_array])

    # Normalize color
    input_array = ColorNormalize(input_array)

    # Add batch dimension and convert to torch tensor
    input_array = np.expand_dims(input_array, axis=0)  # Add batch dimension
    input_array = np.expand_dims(input_array, axis=0)  # Add frames dimension
    
    # Convert array to torch tensor
    preprocessed_input = torch.from_numpy(input_array).float()
    return preprocessed_input

def ColorNormalize(batch_img):
    mean = 0.413621
    std = 0.1700239
    batch_img = (batch_img - mean) / std
    return batch_img
def predict_single_output(model, test_data, use_gpu=False):
    model.eval()  # Set the model to evaluation mode
    with torch.no_grad():
        if use_gpu:
            test_data = test_data.cuda()
        output = model(test_data)
        _, predicted_label = torch.max(output, 1)
    return predicted_label.item()  # Return the predicted label as a scalar

if __name__ == '__main__':
    # Load the trained model
    saved_model = Lipreading(mode=args.mode, inputDim=512, hiddenDim=1024, nClasses=44, frameLen=30, every_frame=args.every_frame)
    saved_model.load_state_dict(torch.load('modelnew44.pth'))

    # Determine if GPU is available
    use_gpu = torch.cuda.is_available()

    # Load and preprocess the input data
    input_data_path = r"C:\Users\ml\Desktop\speakerIndependent\EKMOT\test\EKMOT_005.npy"  # File path to the input data
    input_data = preprocess_input(input_data_path)

    # Predict the single output
    predicted_label = predict_single_output(saved_model, input_data, use_gpu)

    # Print the predicted label
    print("Predicted label:", predicted_label)